# SQL
> #### Tooling for querying SQLite database.

## Setup

### Importing the necessary libraries

In [13]:
from os import getenv
from dotenv import load_dotenv

from IPython.display import Markdown

from langchain.tools import tool
from langchain_community.utilities import SQLDatabase

from langchain.agents import create_agent
from langchain.messages import HumanMessage

### Environment config.

In [3]:
load_dotenv()

OLLAMA_MODEL = getenv("OLLAMA_MODEL")
DB_URI = "sqlite:///resources/Chinook.db"

### DB setup

In [8]:
db = SQLDatabase.from_uri(DB_URI)
print(db._all_tables)

{'Invoice', 'Album', 'Customer', 'Employee', 'Genre', 'PlaylistTrack', 'InvoiceLine', 'MediaType', 'Track', 'Playlist', 'Artist'}


## Agent setup

### Query tool

In [ ]:
@tool
def sql_query(query: str) -> str:
    """Fetch relevant information from the SQLite DB using SQL queries"""
    try:
        return db.run(query)  # type: ignore
    except Exception as e:
        return f"Error: {e}"


result = sql_query.invoke("SELECT * FROM Artist LIMIT 10")
pprint(result)

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [10]:
agent = create_agent(
    model=OLLAMA_MODEL,
    tools=[sql_query]
)

In [11]:
question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [14]:
Markdown(response["messages"][-1].content)

The most popular artist beginning with 'S' in the database is the **Smashing Pumpkins**.

In [16]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT ar.Name 
FROM artist ar 
JOIN album al ON ar.ArtistId = al.ArtistId 
JOIN track t ON al.AlbumId = t.AlbumId 
WHERE ar.Name LIKE 'S%' 
GROUP BY ar.ArtistId 
ORDER BY COUNT(t.TrackId) DESC 
LIMIT 1;
